# Linear Regression

## Load the Dataset

In [2]:
import pandas as pd

exploratory_df = pd.read_csv("dataset/processed/exploratory-packages-topic-modeling.csv")
confirmatory_df = pd.read_csv("dataset/processed/confirmatory-packages-topic-modeling.csv")
holdout_df = pd.read_csv("dataset/processed/holdout-packages-topic-modeling.csv")

exploratory_df.head()

,package_id,created_at,test_week,clickability_test_id,headline,eyecatcher_id,impressions,clicks,ctr,ctr_demeaned,...,neu,pos,compound,created_at_dayofweek,created_at_hourofday,test_group_size,read_flesch,read_coleman,specificity_tfidf,topic_bertopic
0,62,2014-11-20 14:57:52.478,201446,546e009a9ad54ec65b00004b,What They Learned From The Scientist Was Terri...,546c7f2dbadeb5788700000a,4594,51,0.011101,0.111516,...,0.856,0.000,-0.3291,3,14,6,80.782500,8.093333,0.444616,4
1,84,2014-11-20 14:54:18.780,201446,546e009a9ad54ec65b00004b,A Science Guy Helps 3 Dudes From America Under...,546c7f2dbadeb5788700000a,4571,58,0.012689,0.682108,...,0.809,0.191,0.3818,3,14,6,59.682143,10.257143,0.370851,4
2,95,2014-11-20 15:04:49.517,201446,546e009a9ad54ec65b00004b,He Sat Them Down And Told Them About An Immine...,546c7f2dbadeb5788700000a,4601,27,0.005868,-1.769714,...,0.813,0.000,-0.5994,3,15,6,80.465000,6.077778,0.398801,-1
3,100,2014-11-20 15:13:36.266,201446,546e009a9ad54ec65b00004b,"The 3 Of Them Needed To See It In Person, And ...",546c7f2dbadeb5788700000a,4567,63,0.013795,1.079669,...,0.720,0.156,0.2023,3,15,6,80.777143,3.228571,0.445514,-1
4,102,2014-11-20 15:15:25.697,201446,546e009a9ad54ec65b00004b,"They May Not Be The Most Handsome Dudes, But T...",546c7f2dbadeb5788700000a,4524,44,0.009726,-0.382964,...,0.655,0.345,0.7812,3,15,6,92.965000,5.875000,0.497006,-1


### Basic Linear Regression

First I'll run a basic linear regression model to predict the click rate from the features. All the numerical features will be used as predictors.

This will be affected by severe multicollinearity but it's a good starting point.

In [3]:
exploratory_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12010 entries, 0 to 12009
Data columns (total 51 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   package_id                     12010 non-null  int64  
 1   created_at                     12010 non-null  object 
 2   test_week                      12010 non-null  int64  
 3   clickability_test_id           12010 non-null  object 
 4   headline                       12010 non-null  object 
 5   eyecatcher_id                  11996 non-null  object 
 6   impressions                    12010 non-null  int64  
 7   clicks                         12010 non-null  int64  
 8   ctr                            12010 non-null  float64
 9   ctr_demeaned                   12010 non-null  float64
 10  first_place                    12010 non-null  bool   
 11  winner                         12010 non-null  bool   
 12  is_highest_ctr                 12010 non-null 

In [4]:
def prepare_data(df):
    # Transform int columns to float
    df["headline_num_persons"] = df["headline_num_persons"].astype(float)
    df["headline_num_orgs"] = df["headline_num_orgs"].astype(float)
    df["headline_num_gpes"] = df["headline_num_gpes"].astype(float)
    df["num_pronouns"] = df["num_pronouns"].astype(float)
    df["num_chars"] = df["num_chars"].astype(float)
    df["num_tokens"] = df["num_tokens"].astype(float)
    df["num_nouns"] = df["num_nouns"].astype(float)
    df["num_verbs"] = df["num_verbs"].astype(float)
    df["num_adjs"] = df["num_adjs"].astype(float)
    df["num_advs"] = df["num_advs"].astype(float)
    df["test_group_size"] = df["test_group_size"].astype(float)

    # Drop rows with missing values in any column
    df = df.dropna()
    return df

exploratory_df = prepare_data(exploratory_df)
confirmatory_df = prepare_data(confirmatory_df)
holdout_df = prepare_data(holdout_df)


In [5]:
exploratory_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11996 entries, 0 to 12009
Data columns (total 51 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   package_id                     11996 non-null  int64  
 1   created_at                     11996 non-null  object 
 2   test_week                      11996 non-null  int64  
 3   clickability_test_id           11996 non-null  object 
 4   headline                       11996 non-null  object 
 5   eyecatcher_id                  11996 non-null  object 
 6   impressions                    11996 non-null  int64  
 7   clicks                         11996 non-null  int64  
 8   ctr                            11996 non-null  float64
 9   ctr_demeaned                   11996 non-null  float64
 10  first_place                    11996 non-null  bool   
 11  winner                         11996 non-null  bool   
 12  is_highest_ctr                 11996 non-null  bool

I'll use Python Statsmodels library

In [6]:
import statsmodels.api as sm

def basic_linear_regression(df):
    target_col = "ctr_demeaned"
    feature_cols = [
        "headline_num_persons",
        "headline_num_orgs",
        "headline_num_gpes",
        "headline_has_person",
        "headline_has_org",
        "headline_has_gpe",
        "headline_has_money",
        "starts_with_verb",
        "starts_with_pronoun",
        "starts_with_number",
        "num_pronouns",
        "num_chars",
        "num_tokens",
        "avg_token_len",
        "ends_with_qmark",
        "ends_with_exclaim",
        "has_quote",
        "has_all_caps_word",
        "num_nouns",
        "num_verbs",
        "num_adjs",
        "num_advs",
        "headline_imperative_verb",
        "headline_has_curiosity_word",
        "headline_curiosity_similarity",
        "headline_has_intensity_word",
        "headline_intensity_similarity",
        "neg",
        "neu",
        "pos",
        "compound",
        "test_group_size",
        "read_flesch",
        "read_coleman",
        "specificity_tfidf",
    ]

    # Drop rows with missing values in any of these columns
    model_df = df[feature_cols + [target_col]].dropna()

    X = model_df[feature_cols]
    y = model_df[target_col]

    # Ensure all predictors and target are numeric (convert bools to 0/1 and ints to float)
    X = X.astype(float)
    y = y.astype(float)

    # Add intercept term
    X = sm.add_constant(X)

    # Fit OLS regression using Statsmodels
    return sm.OLS(y, X).fit()

ols_model = basic_linear_regression(exploratory_df)
# Display regression summary
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:           ctr_demeaned   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     5.991
Date:                Sun, 14 Dec 2025   Prob (F-statistic):           1.70e-26
Time:                        17:27:08   Log-Likelihood:                -15450.
No. Observations:               11996   AIC:                         3.097e+04
Df Residuals:                   11960   BIC:                         3.124e+04
Df Model:                          35                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

R-squared is very low, which means that only 5% of the variance is explained by the linear regression. This is expected, we don't think a linear regression over the engineered features will help us to certainly predict the click rate, but it's valueable to see which features are more important and could give us some insights.

The features that seem more significant (lower p-values and a confidence interval that doesn't include 0) are:
- headline_num_persons (0.007) -> People with names is mentioned in the headline, increases the click rate
- headline_has_money (0.001) -> Headlines that mention money are less effective
- num_chars (0.083) and num_tokens (0.005) -> Slightly decrease the click rate
- ends_with_qmark and ends_with_exclaim (0.000) -> Decrease the click rate
- num_nouns (0.000) -> Increases the click rate
- compound (0.000) -> Decreases the click rate, which means more positive sentiment in the headline decreases the click rate
- test_group_size (0.000) -> Larger test groups have lower click rates, this may be expected because, with larger test groups, headlines with poorer performance are more likely to be shown
- specificity_tfidf (0.000) -> More specificity decreases the click rate, which means that headlines with more unique words are less effective


This tell us with a minimal statistical significance that audience prefer to click on headlines that:
- Mention people by name and don't talk about money
- Are shorter rather that larger
- Aren't a question or an exclamation
- Have more nouns (This may be related to the mention of people)
- Have a more neutral or negative sentiment
- Have less specific words

All this makes sense, people prefers simpler headlines, more negative, and it's better if they talk about someone specifically.




## Analyze individual features

Now we'll run linear regressions to analyze the effect of specific changes in the headlines, and avoid the multicollinearity issue of fitting all the features at once.

These are the groups of features that will be fitted:

- Named entities: headline_num_persons, headline_num_orgs, headline_num_gpes
- Named entities boolean: headline_has_person, headline_has_org, headline_has_gpe, headline_has_money
- Sentence starting with: starts_with_verb, starts_with_pronoun, starts_with_number
- Basic text features: num_pronouns, num_nouns, num_verbs, num_adjs, num_advs, num_chars, num_tokens, avg_token_len
- Question/Exclamation features: ends_with_qmark, ends_with_exclaim, has_quote, has_all_caps_word",
- Sentence style: headline_imperative_verb, headline_has_curiosity_word, headline_curiosity_similarity, headline_has_intensity_word, headline_intensity_similarity
- Sentiment: neg, neu, pos, compound
- Test size: test_group_size
- Readability and specificity: read_flesch, read_coleman, specificity_tfidf

In [8]:
def feature_groups_regression(df):

    target_col = "ctr_demeaned"

    feature_groups = {
        "named_entities": [
            "headline_num_persons",
            "headline_num_orgs",
            "headline_num_gpes",
        ],
        "named_entities_bool": [
            "headline_has_person",
            "headline_has_org",
            "headline_has_gpe",
            "headline_has_money",
        ],
        "sentence_start": [
            "starts_with_verb",
            "starts_with_pronoun",
            "starts_with_number",
        ],
        "basic_text_features": [
            "num_pronouns",
            "num_nouns",
            "num_verbs",
            "num_adjs",
            "num_advs",
            "num_chars",
            "num_tokens",
            "avg_token_len",
        ],
        "question_exclaim_features": [
            "ends_with_qmark",
            "ends_with_exclaim",
            "has_quote",
            "has_all_caps_word",
        ],
        "sentence_style": [
            "headline_imperative_verb",
            "headline_has_curiosity_word",
            "headline_curiosity_similarity",
            "headline_has_intensity_word",
            "headline_intensity_similarity",
        ],
        "sentiment": [
            "neg",
            "neu",
            "pos",
            "compound",
        ],
        "test_size": [
            "test_group_size",
        ],
        "readability_specificity": [
            "read_flesch",
            "read_coleman",
            "specificity_tfidf",
        ],
    }

    group_models = {}

    for group_name, cols in feature_groups.items():
        print("\n")
        print(f"Group: {group_name} — features: {cols}")
        print("=================\n")

        # Subset data and drop rows with missing values
        model_df = df[cols + [target_col]].dropna()

        X = model_df[cols].astype(float)
        y = model_df[target_col].astype(float)

        X = sm.add_constant(X)

        model = sm.OLS(y, X).fit()
        group_models[group_name] = model

        print(model.summary())

feature_groups_regression(exploratory_df)



Group: named_entities — features: ['headline_num_persons', 'headline_num_orgs', 'headline_num_gpes']

                            OLS Regression Results                            
Dep. Variable:           ctr_demeaned   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.5115
Date:                Sun, 14 Dec 2025   Prob (F-statistic):              0.674
Time:                        17:29:36   Log-Likelihood:                -15553.
No. Observations:               11996   AIC:                         3.111e+04
Df Residuals:                   11992   BIC:                         3.114e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
-----------------

**Named entities**

The mention of people clearly increases the CTR while the mention of money decreases it.

**Sentence starting**

Headlines starting with a pronoun are associated with a lower CTR, this isn't something we expected.

**Basic text features**

The only conclusion we can draw from this is that users prefer shorter headlines in general.

**Question/Exclamation features**

Headlines with questions, exclamation marks, or all caps words have clearly a lower CTR.

**Sentence style**

Not much can be interpreted from this, statistically headlines are not affected by the use of curiosity or intensity words, as well as the use of verbs in imperative modes. I expected something different from this. I would have assumed that the use of curiosity words would increase the CTR.

**Sentiment**

Compound ranges from -1 to 1, with 0 being neutral, -1 negative and 1 positive. Positiveness is associated with a decrease in the CTR.

**Test Size**

It can't be denied that the use of multiple headlines (packages) in an A/B test decreases the CTR overall. This may be because the test of more headlines, include more versions of the headlines that aren't so effective.

**Specificity**

The use of specific words is associated with a decrease in the CTR.

In general, the regression of the features independently gave us the same results as the evaluation of all the features simultaneously.

## Hypothesis Confirmation

Now I'll run the same model on the confirmatory dataset to evaluate if the same correlations hold.

In [9]:
feature_groups_regression(confirmatory_df)



Group: named_entities — features: ['headline_num_persons', 'headline_num_orgs', 'headline_num_gpes']

                            OLS Regression Results                            
Dep. Variable:           ctr_demeaned   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     9.113
Date:                Sun, 14 Dec 2025   Prob (F-statistic):           5.02e-06
Time:                        17:32:45   Log-Likelihood:                -74292.
No. Observations:               57324   AIC:                         1.486e+05
Df Residuals:                   57320   BIC:                         1.486e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
-----------------

- **Named entities**: Valid for the confirmatory dataset too
- **Sentence starting**: Valid for the confirmatory dataset too
- **Basic text features**: Valid for the confirmatory dataset too
- **Question/Exclamation features**: Valid for the confirmatory dataset too
- **Sentence style**: There is a slight decrease in the CTR for headlines with words related to curiosity
- **Sentiment**: Here the correlation is not statistically significant
- **Test Size**: Valid for the confirmatory dataset too
- **Readability and specificity**: Valid for the confirmatory dataset too